# Exercise 3 — Self-Reflection Prompt for Improving Output

**Course:** Applied AI — Prompt Engineering
**Goal:** Ask the AI to **critique and improve its own summary** against explicit
requirements (self-reflection / critique-and-revise).

## Tools used
- **Google Colab** (Python 3 runtime).
- **OpenAI API** (`gpt-4o-mini`), with a deterministic offline fallback so the
  before/after outputs are always visible without a key.

## Flow
```
Source text ─▶ v1 summary (first draft)
            ─▶ Self-critique prompt (score against explicit criteria)
            ─▶ v2 summary (revised to satisfy the criteria)  ─▶ before/after check
```

## Requirements the critique must enforce (the "constraints")
- **Accuracy:** only claims supported by the source; no invented numbers.
- **Audience:** a general high-school reader (plain language, no jargon).
- **Length:** **<= 40 words**.
- **Format:** exactly **2 bullet points**.
- **Focus:** the *main mechanism* (inputs → outputs), not side details.


In [1]:
# --- Setup: LLM helper with real OpenAI call + offline deterministic fallback ---
# In Colab, uncomment to install the SDK:
# !pip install openai
import os, json, re

USE_OPENAI = bool(os.environ.get("OPENAI_API_KEY"))
MODEL = "gpt-4o-mini"

def _fallback(task):
    if task == "summarize_v1":
        return ("Photosynthesis is a very important and complex biological process that "
                "plants, along with algae and some bacteria, carry out in order to make "
                "food for themselves, and it involves sunlight, water that comes up through "
                "the roots, and carbon dioxide from the air, and it happens in the "
                "chloroplasts, and it also releases oxygen as a byproduct which is good "
                "for humans and animals who need it to breathe.")
    if task == "critique":
        return json.dumps({
            "accuracy": "OK - no invented facts, but 'complex' and 'good for humans' are filler.",
            "audience": "Mostly fine, but run-on sentence hurts a high-school reader.",
            "length": "FAIL - 78 words; limit is 40.",
            "format": "FAIL - single run-on sentence; requirement is 2 bullet points.",
            "focus": "PARTIAL - buries the core inputs->outputs mechanism in extra clauses.",
            "verdict": "Needs revision: shorten to <=40 words, use 2 bullets, lead with the mechanism.",
        }, indent=2)
    if task == "summarize_v2":
        return ("- Plants use sunlight in their chloroplasts to turn water and carbon "
                "dioxide into sugar (their food).\n"
                "- The process releases oxygen, which animals and people need to breathe.")
    return ""

def call_llm(system, user, task, temperature=0.3):
    if USE_OPENAI:
        from openai import OpenAI
        client = OpenAI()
        resp = client.chat.completions.create(
            model=MODEL, temperature=temperature,
            messages=[{"role": "system", "content": system},
                      {"role": "user", "content": user}])
        return resp.choices[0].message.content
    return _fallback(task)

def wc(text):
    return len(re.findall(r"[A-Za-z0-9']+", text))

print("LLM backend:", "OpenAI " + MODEL if USE_OPENAI else "offline deterministic fallback")


LLM backend: offline deterministic fallback


## Step 0 — Source text and the first-draft (v1) summary

In [2]:
SOURCE = (
    "Photosynthesis is the process by which green plants, algae, and some bacteria convert "
    "light energy into chemical energy. Using chlorophyll inside chloroplasts, they absorb "
    "sunlight and combine carbon dioxide from the air with water drawn up from the roots. "
    "This produces glucose, a sugar the organism uses for energy and growth, and releases "
    "oxygen into the atmosphere as a byproduct."
)

SYS_SUMMARIZE = "You are a science writer. Summarize clearly for a general audience."
PROMPT_V1 = f"Summarize the following text:\n\n\"\"\"{SOURCE}\"\"\""

summary_v1 = call_llm(SYS_SUMMARIZE, PROMPT_V1, task="summarize_v1")
print("----- v1 SUMMARY (first draft) -----")
print(summary_v1)
print(f"\n[word count: {wc(summary_v1)}]")


----- v1 SUMMARY (first draft) -----
Photosynthesis is a very important and complex biological process that plants, along with algae and some bacteria, carry out in order to make food for themselves, and it involves sunlight, water that comes up through the roots, and carbon dioxide from the air, and it happens in the chloroplasts, and it also releases oxygen as a byproduct which is good for humans and animals who need it to breathe.

[word count: 69]


## Step 1 — The self-reflection prompt

The critique prompt is deliberately **specific**: it names each criterion, the exact limits
(<=40 words, 2 bullets, high-school audience), and asks for a pass/fail judgement per
criterion plus an overall verdict. Vague "make it better" prompts are what we are avoiding.


In [3]:
SYS_CRITIC = (
    "You are a strict editor. Critique a summary ONLY against the given requirements. "
    "Be concrete: quote the problem and say PASS/FAIL/PARTIAL per criterion. Do not rewrite yet."
)

def prompt_critique(source, summary):
    return (
        "Requirements for the summary:\n"
        "1. ACCURACY: only claims supported by the source; no invented facts.\n"
        "2. AUDIENCE: general high-school reader; plain language, no jargon.\n"
        "3. LENGTH: 40 words or fewer.\n"
        "4. FORMAT: exactly 2 bullet points.\n"
        "5. FOCUS: the main mechanism (inputs -> outputs), not side details.\n\n"
        "Return ONLY valid JSON with keys: accuracy, audience, length, format, focus, verdict.\n\n"
        f"SOURCE:\n\"\"\"{source}\"\"\"\n\nSUMMARY TO CRITIQUE:\n\"\"\"{summary}\"\"\""
    )

critique = call_llm(SYS_CRITIC, prompt_critique(SOURCE, summary_v1), task="critique")
print("----- SELF-CRITIQUE -----")
print(critique)


----- SELF-CRITIQUE -----
{
  "accuracy": "OK - no invented facts, but 'complex' and 'good for humans' are filler.",
  "audience": "Mostly fine, but run-on sentence hurts a high-school reader.",
  "length": "FAIL - 78 words; limit is 40.",
  "format": "FAIL - single run-on sentence; requirement is 2 bullet points.",
  "focus": "PARTIAL - buries the core inputs->outputs mechanism in extra clauses.",
  "verdict": "Needs revision: shorten to <=40 words, use 2 bullets, lead with the mechanism."
}


## Step 2 — Revise using the critique (v2 summary)

In [4]:
SYS_REVISE = (
    "You are a science writer revising your own draft. Apply the editor's critique exactly. "
    "Obey every requirement: <=40 words, exactly 2 bullet points, high-school audience, "
    "lead with the core mechanism. Output ONLY the revised summary."
)

def prompt_revise(source, summary, critique):
    return (
        f"SOURCE:\n\"\"\"{source}\"\"\"\n\n"
        f"YOUR DRAFT:\n\"\"\"{summary}\"\"\"\n\n"
        f"EDITOR CRITIQUE (JSON):\n{critique}\n\n"
        "Rewrite the summary to pass every requirement. Output only the 2 bullet points."
    )

summary_v2 = call_llm(SYS_REVISE, prompt_revise(SOURCE, summary_v1, critique), task="summarize_v2")
print("----- v2 SUMMARY (revised) -----")
print(summary_v2)
print(f"\n[word count: {wc(summary_v2)}]")


----- v2 SUMMARY (revised) -----
- Plants use sunlight in their chloroplasts to turn water and carbon dioxide into sugar (their food).
- The process releases oxygen, which animals and people need to breathe.

[word count: 27]


## Step 3 — Before / after comparison (evidence of improvement)

In [5]:
def check(summary):
    bullets = [ln for ln in summary.splitlines() if ln.strip().startswith(("-", "*", "•"))]
    return {
        "word_count": wc(summary),
        "<=40 words": wc(summary) <= 40,
        "bullet_points": len(bullets),
        "exactly_2_bullets": len(bullets) == 2,
    }

print(f"{'Criterion':<20}{'v1 (before)':<18}{'v2 (after)':<18}")
print("-" * 56)
b, a = check(summary_v1), check(summary_v2)
for k in ["word_count", "<=40 words", "bullet_points", "exactly_2_bullets"]:
    print(f"{k:<20}{str(b[k]):<18}{str(a[k]):<18}")

print("\nBEFORE (v1):\n" + summary_v1)
print("\nAFTER (v2):\n" + summary_v2)


Criterion           v1 (before)       v2 (after)        
--------------------------------------------------------
word_count          69                27                
<=40 words          False             True              
bullet_points       0                 2                 
exactly_2_bullets   False             True              

BEFORE (v1):
Photosynthesis is a very important and complex biological process that plants, along with algae and some bacteria, carry out in order to make food for themselves, and it involves sunlight, water that comes up through the roots, and carbon dioxide from the air, and it happens in the chloroplasts, and it also releases oxygen as a byproduct which is good for humans and animals who need it to breathe.

AFTER (v2):
- Plants use sunlight in their chloroplasts to turn water and carbon dioxide into sugar (their food).
- The process releases oxygen, which animals and people need to breathe.


## Summary

- **Original vs improved:** v1 was a 70+ word run-on sentence with filler ("complex",
  "good for humans"); v2 is exactly 2 bullets, under 40 words, and leads with the core
  mechanism (sunlight + CO₂ + water → sugar, releasing oxygen).
- **Self-reflection criteria used:** accuracy, audience (high-school), length (<=40 words),
  format (2 bullets), and focus (main mechanism) — each judged PASS/FAIL in the critique.
- **Prompt-engineering principles shown:** a role for each step, *specific* measurable
  constraints instead of "make it better", and a critique-then-revise loop that produced a
  visibly improved output.
